In [1]:

import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import resnet50
from sklearn.model_selection import train_test_split

# FX Quantization imports
from torch.ao.quantization import get_default_qconfig, QConfigMapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx


# Backend & device

torch.backends.quantized.engine = "fbgemm"   # x86 CPU INT8 backend
DEVICE = torch.device("cpu")                 # PTQ evaluate/convert on CPU


#  Dataset & DataLoaders


transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"   # change if needed
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
NUM_CLASSES = len(dataset.classes)

print(f"Total images: {len(dataset)}, Classes: {NUM_CLASSES}")

# Stratified split 85/10/5 (train/val/test)
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for cid in np.unique(targets):
    idx = np.where(targets == cid)[0]
    tr, tmp = train_test_split(idx, test_size=0.15, random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3, random_state=42, shuffle=True)  # -> 10%/5%
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")


# Build FP32 model & load weights

model_fp32 = resnet50(weights=None)
model_fp32.fc = nn.Linear(model_fp32.fc.in_features, NUM_CLASSES)

ckpt_path = "resnet50_weights.pth"  # path to weights saved by your baseline
state = torch.load(ckpt_path, map_location="cpu")

# strip _orig_mod. if present (common when saving from torch.compile)
if any(k.startswith("_orig_mod.") for k in state.keys()):
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}

missing, unexpected = model_fp32.load_state_dict(state, strict=False)
print("Missing keys:", list(missing))
print("Unexpected keys:", list(unexpected))
model_fp32.eval().to(DEVICE)
print("FP32 ResNet-50 loaded")


# FX Static PTQ

qconfig = get_default_qconfig("fbgemm")  # per-tensor activations, per-channel weights where supported
qconfig_mapping = QConfigMapping().set_global(qconfig)

example_inputs = (torch.randn(1, 3, 224, 224),)
prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

# Calibration with a few loader batches
def calibrate_fx(m, loader, num_batches=20):
    m.eval()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            m(images.to("cpu"))
calibrate_fx(prepared, train_loader, num_batches=20)
print("Calibration done")

# Convert to INT8
model_int8 = convert_fx(prepared).to("cpu").eval()
print("Static quantization complete")


#  Evaluation

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            # FX int8 models accept float inputs (they contain Q/DQ nodes internally)
            outputs = model(images.to("cpu"))
            preds = outputs.argmax(1).cpu()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

acc_fp32 = evaluate(model_fp32, test_loader)
acc_int8 = evaluate(model_int8, test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy (Static PTQ): {acc_int8:.2f}%")


# Model size & Runtime

def get_model_size(model, filename="__tmp__.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    t0 = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(images.to("cpu"))
    t1 = time.time()
    imgs = num_batches * loader.batch_size
    return (t1 - t0) / num_batches * 1000.0, (t1 - t0) / imgs * 1000.0  # ms/batch, ms/image

fp32_size = get_model_size(model_fp32)
int8_size = get_model_size(model_int8)
print(f"FP32 model size: {fp32_size:.2f} MB | INT8 model size: {int8_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_int8, img_ms_int8 = benchmark(model_int8, test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"INT8 Inference: {batch_ms_int8:.2f} ms/batch | {img_ms_int8:.2f} ms/image")



Total images: 100000, Classes: 200
DataLoaders ready
Missing keys: []
Unexpected keys: []
FP32 ResNet-50 loaded


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/ao/quantization/observer.py:220: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Calibration done
Static quantization complete
FP32 Accuracy: 73.42%
INT8 Accuracy (Static PTQ): 72.92%
FP32 model size: 95.98 MB | INT8 model size: 24.51 MB
FP32 Inference: 3928.10 ms/batch | 30.69 ms/image
INT8 Inference: 2744.69 ms/batch | 21.44 ms/image
